# HallucinationMetric

## What it measures

The proportion of supplied `context` documents that the answer **contradicts**. This is a
rate: `0.0` is perfect and higher is worse, and the metric passes when the score is at or
below the threshold. It is the inverse of Faithfulness, and it uses `context` - curated
ground truth - rather than `retrieval_context`.

That distinction matters. Faithfulness asks "is the answer supported by what was
retrieved?", which will happily pass an answer faithfully derived from a bad retrieval.
Hallucination asks "does the answer contradict what is actually true?", which is the
question a compliance reviewer cares about.

## When it is useful

On any surface where a confident wrong statement is expensive - here, an assistant telling
an analyst the reporting threshold is GBP 15,000 when policy says GBP 10,000. It is the
metric to run after a model swap, when fluency is unchanged but grounding may not be.

## DeepEval inputs and test-case type

| DeepEval field | Required |
|---|---|
| test case type | `LLMTestCase` |
| `input` | yes |
| `actual_output` | yes |
| `context` | yes - **ground truth**, not live retrieval output |

In [ ]:
# --------------------------------------------------------------------------
# Configuration. Every value comes from the environment - nothing about this
# machine, this port or this deployment is baked into the notebook.
# --------------------------------------------------------------------------
import json
import os
import textwrap
from pathlib import Path

import httpx
from dotenv import load_dotenv

# Look for .env next to the notebook, then one level up (the project root).
for _candidate in (Path.cwd() / ".env", Path.cwd().parent / ".env"):
    if _candidate.is_file():
        load_dotenv(_candidate)
        break


class MissingConfiguration(RuntimeError):
    """Raised when a required environment variable is absent."""


def env(name, default=None, *, required=False):
    value = os.environ.get(name) or default
    if required and not value:
        raise MissingConfiguration(
            f"Environment variable {name!r} is not set.\n"
            f"Copy .env.example to .env and fill it in, or export {name} before "
            f"starting the kernel. See README.md -> '.env configuration'."
        )
    return value


API_BASE = env("AML_API_BASE_URL", "http://localhost:8000").rstrip("/")
API_TIMEOUT_S = float(env("AML_API_TIMEOUT_S", "180"))
EXPECTED_SEED_VERSION = env("AML_EXPECTED_SEED_VERSION", "scenarios-v1")
RESET_BEFORE_RUN = env("AML_RESET_BEFORE_RUN", "false").lower() in ("1", "true", "yes")

# Every notebook needs an OpenAI key. ToolCorrectnessMetric scores without any
# LLM call, but DeepEval 4.1.4 still builds a GPTModel in its constructor and
# raises without a key, so the key is required there too - just never used.
JUDGE_MODEL = env("DEEPEVAL_JUDGE_MODEL", "gpt-5.4-mini")
os.environ.setdefault("DEEPEVAL_TELEMETRY_OPT_OUT", "YES")

# One key per role, falling back to the single AML_API_KEY. Blank is correct
# when the application runs with AUTH_MODE=off (its default).
API_KEYS = {
    "analyst": env("AML_API_KEY_ANALYST") or env("AML_API_KEY", ""),
    "eval_reader": env("AML_API_KEY_EVAL_READER") or env("AML_API_KEY", ""),
    "test_operator": env("AML_API_KEY_TEST_OPERATOR") or env("AML_API_KEY", ""),
}

print(f"API base URL       : {API_BASE}")
print(f"Request timeout    : {API_TIMEOUT_S}s")
print(f"Judge model        : {JUDGE_MODEL}")
print(f"Expected seed      : {EXPECTED_SEED_VERSION}")
print(f"API key configured : {bool(API_KEYS['analyst'])}  (False is correct when AUTH_MODE=off)")
print(f"OPENAI_API_KEY set : {bool(os.environ.get('OPENAI_API_KEY'))}")

In [ ]:
# --------------------------------------------------------------------------
# A small HTTP client. Every failure mode the application can present is
# turned into a message that names the cause and the thing to check.
# --------------------------------------------------------------------------
EXPECTED_SCHEMA_VERSION = "1.0.0"

SECRET_KEY_HINTS = ("api_key", "apikey", "authorization", "secret", "password",
                    "credential", "token")


def redact(value):
    """Mask credential-like values before anything is printed."""
    if isinstance(value, dict):
        return {
            k: ("***REDACTED***" if any(h in k.lower() for h in SECRET_KEY_HINTS)
                else redact(v))
            for k, v in value.items()
        }
    if isinstance(value, list):
        return [redact(v) for v in value]
    return value


class ApiError(RuntimeError):
    """A non-2xx response, carrying the application's error envelope."""


def api(method, path, *, role="analyst", json_body=None, params=None,
        expect_status=None):
    """Call the application API and return parsed JSON.

    role selects which API key is sent. It only matters when the application
    runs with AUTH_MODE=api_key; with AUTH_MODE=off the header is omitted.
    """
    headers = {"Accept": "application/json"}
    key = API_KEYS.get(role, "")
    if key:
        headers["X-API-Key"] = key

    url = f"{API_BASE}{path}"
    try:
        response = httpx.request(method, url, headers=headers, json=json_body,
                                 params=params, timeout=API_TIMEOUT_S)
    except httpx.ConnectError as exc:
        raise ApiError(
            f"Could not connect to {url}.\n"
            f"  - Is the application running?  curl {API_BASE}/api/health\n"
            f"  - Is AML_API_BASE_URL correct? It is currently {API_BASE!r}.\n"
            f"  - Underlying error: {exc}"
        ) from exc
    except httpx.TimeoutException as exc:
        raise ApiError(
            f"{method} {url} timed out after {API_TIMEOUT_S}s.\n"
            f"  - An investigation run does retrieval, several MCP tool calls and\n"
            f"    one LLM synthesis; raise AML_API_TIMEOUT_S if this is expected.\n"
            f"  - Underlying error: {exc!r}"
        ) from exc

    served = response.headers.get("X-Schema-Version")
    if served and served != EXPECTED_SCHEMA_VERSION:
        print(f"WARNING: application reports contract version {served}, these "
              f"notebooks were written against {EXPECTED_SCHEMA_VERSION}. "
              f"Field names may have changed - see docs/evaluation-contract.md.")

    if response.status_code >= 400:
        try:
            envelope = response.json()
        except ValueError:
            envelope = {"raw_body": response.text[:1000]}
        hint = {
            401: "AUTH_MODE=api_key is on and no valid X-API-Key was sent. Set AML_API_KEY.",
            403: "The key's role may not reach this endpoint. eval_reader is needed for "
                 "/api/agent/trace and /api/eval/*; test_operator for /api/dev/reset and "
                 "/api/mcp/invoke.",
            404: "The id does not exist. Resolve ids from GET /api/eval/scenarios rather "
                 "than hardcoding them.",
            409: "Often index_not_built - the vector index has never been built. "
                 "POST /api/dev/reset once, or set AML_RESET_BEFORE_RUN=true.",
            502: "The application's LLM provider failed or returned output that broke its "
                 "own schema contract. Retry, or inspect GET /api/agent/trace/{run_id}.",
            503: "llm_not_configured - the application has no OPENROUTER_API_KEY. "
                 "This is the application's key, not the judge's OPENAI_API_KEY.",
        }.get(response.status_code, "")
        raise ApiError(
            f"{method} {url} -> HTTP {response.status_code}\n"
            f"  envelope: {json.dumps(envelope, indent=2)[:1200]}\n"
            + (f"  hint: {hint}" if hint else "")
        )

    if expect_status is not None and response.status_code != expect_status:
        raise ApiError(f"{method} {url} -> expected HTTP {expect_status}, "
                       f"got {response.status_code}")

    if not response.content:
        return None
    try:
        return response.json()
    except ValueError as exc:
        raise ApiError(
            f"{method} {url} returned HTTP {response.status_code} but the body is not "
            f"JSON.\n  first 500 bytes: {response.text[:500]!r}"
        ) from exc


def show(title, payload, limit=2500):
    """Pretty-print a payload with secrets masked and long bodies truncated."""
    text = json.dumps(redact(payload), indent=2, default=str)
    print(f"----- {title} -----")
    print(text if len(text) <= limit else text[:limit] + f"\n... [{len(text) - limit} more characters]")


health = api("GET", "/api/health")
show("GET /api/health", health)
if not health.get("status") == "ok":
    raise ApiError(f"Application is not healthy: {health}")

## Endpoint exercised

Two endpoints, for two different roles:

- `POST /api/rag/query` produces the `actual_output` under test.
- `GET /api/documents/{document_id}` supplies the `context`: the **full markdown text** of
  the governing policy documents, as the application itself serves them.

Using the full documents rather than the retrieved chunks is the point. The retrieved
chunks are a subset the retriever chose; if retrieval missed the clause that contradicts
the answer, a chunk-based context would score the contradiction as fine. The application's
own golden-format specification says the same thing - `context` is "curated ground truth
for HallucinationMetric only. Not live retrieval output." 

In [ ]:
# --------------------------------------------------------------------------
# Resolve scenarios to live row ids. Seed ids are assigned by insert order, so
# a hardcoded case_id silently rebinds to a different case when the seed data
# changes. GET /api/eval/scenarios exists precisely to avoid that.
# --------------------------------------------------------------------------
if RESET_BEFORE_RUN:
    # Drops and recreates every table, restoring deterministic seed state.
    reset = api("POST", "/api/dev/reset", role="test_operator")
    show("POST /api/dev/reset", reset)

SCENARIOS = {s["scenario_id"]: s for s in api("GET", "/api/eval/scenarios",
                                              role="eval_reader")}

seed_versions = {s["seed_version"] for s in SCENARIOS.values()}
if seed_versions != {EXPECTED_SEED_VERSION}:
    raise RuntimeError(
        f"Seed version mismatch: application reports {seed_versions}, the goldens in "
        f"this notebook were authored against {EXPECTED_SEED_VERSION!r}.\n"
        f"A golden authored against different seed data is not a weaker test, it is a "
        f"wrong one - fix the seed or the golden rather than lowering the threshold."
    )

for sid, s in sorted(SCENARIOS.items()):
    print(f"{sid}: case_id={s['case_id']} customer_id={s['customer_id']} "
          f"transaction_id={s['transaction_id']}  {s['title']}")

In [ ]:
# --------------------------------------------------------------------------
# The exact request.
# --------------------------------------------------------------------------
CASE_ID = SCENARIOS["s6"]["case_id"]      # "Structuring behaviour" scenario

QUESTION = (
    "What indicators identify structuring, and what cash transaction thresholds "
    "require review under the transaction monitoring policy?"
)

request_body = {
    "question": QUESTION,
    "case_id": CASE_ID,
    "top_k": 8,
    "include_history": False,
}

print("POST", f"{API_BASE}/api/rag/query")
print("headers:", json.dumps(redact({"X-API-Key": API_KEYS["analyst"] or None,
                                     "Content-Type": "application/json"}), indent=2))
print("body:", json.dumps(request_body, indent=2))

In [ ]:
# --------------------------------------------------------------------------
# The raw response.
# --------------------------------------------------------------------------
response = api("POST", "/api/rag/query", json_body=request_body)

show("POST /api/rag/query", {k: v for k, v in response.items()
                             if k != "retrieved_context"})
print()
print("ANSWER")
print(textwrap.fill(response["answer"], width=96, initial_indent="  ",
                    subsequent_indent="  "))

## Mapping the API response onto DeepEval fields

| DeepEval field | Source | Note |
|---|---|---|
| `input` | `question` from the query response | |
| `actual_output` | `answer` | |
| `context` | full `content` of the policy documents from `GET /api/documents/{id}` | Ground truth, independent of what retrieval happened to return |

The response's own `retrieved_context` is **not** used as `context`. It is printed for
debugging so a failure can be attributed to retrieval or to generation, but feeding it in
would make the metric circular.

In [ ]:
# --------------------------------------------------------------------------
# Deriving the ground-truth context.
#
# The application indexes five policy documents and serves each one whole at
# GET /api/documents/{document_id}. Those documents are the authority the
# answer must not contradict, so all of them go in as context - including
# policies the answer does not discuss, which is how a metric of this shape
# catches an answer that strays into a neighbouring policy and gets it wrong.
# --------------------------------------------------------------------------
policy_index = api("GET", "/api/documents", params={"type": "policy"})

context = []
for entry in policy_index:
    document = api("GET", f"/api/documents/{entry['document_id']}")
    context.append(document["content"])
    print(f"context[{len(context) - 1}] <- document_id={document['document_id']} "
          f"{document.get('source')!r} ({len(document['content'])} chars)")

if not context:
    raise RuntimeError(
        "No policy documents were served by GET /api/documents?type=policy, so there is "
        "no ground truth to check the answer against. Seed the application "
        "(POST /api/dev/reset) before running this notebook."
    )

print()
print("For comparison, what retrieval actually returned for this question:")
for chunk in response["retrieved_context"]:
    print(f"  {chunk['chunk_id']:<12} score={chunk['score']:.3f} "
          f"source={chunk.get('source')!r}")

In [ ]:
# --------------------------------------------------------------------------
# Build the test case and print each DeepEval role explicitly.
# --------------------------------------------------------------------------
from deepeval.test_case import LLMTestCase

test_case = LLMTestCase(
    input=response["question"],
    actual_output=response["answer"],
    context=context,
)

print("USER INPUT")
print(" ", test_case.input)
print()
print("ACTUAL OUTPUT")
print(textwrap.fill(test_case.actual_output, width=96, initial_indent="  ",
                    subsequent_indent="  "))
print()
print(f"CONTEXT (ground truth): {len(test_case.context)} whole policy documents")
for n, doc in enumerate(test_case.context):
    print(f"  [{n}] {doc.splitlines()[0][:80]}  ({len(doc)} chars)")
print()
print("RETRIEVAL CONTEXT: deliberately not used - see the mapping section above")

## Judge and threshold

- **Judge model**: `DEEPEVAL_JUDGE_MODEL`, default `gpt-5.4-mini`.
- **Threshold**: `0.5`, DeepEval's documented default for `HallucinationMetric`.

**Direction matters here.** This score is a contradiction *rate*, so lower is better and
the metric succeeds when `score <= threshold`. The result cell prints
`metric.is_successful()` rather than comparing the number by hand, because reading `0.0`
as a failure is the single easiest mistake to make with this metric.

The default `0.5` is retained for consistency with the rest of the suite, but it is a weak
bar for this metric specifically: it tolerates contradicting half the supplied documents.
A real gate on a compliance assistant should sit at `0.0` and treat any contradiction as a
failure. That choice is left to whoever owns the risk, not hardcoded here.

In [ ]:
from deepeval.metrics import HallucinationMetric

metric = HallucinationMetric(
    threshold=0.5,          # DeepEval's documented default; lower is better for this metric
    model=JUDGE_MODEL,
    include_reason=True,
    async_mode=False,
    verbose_mode=True,
)
print(f"metric class : {type(metric).__name__}")
print(f"judge model  : {JUDGE_MODEL}")
print(f"threshold    : {metric.threshold}")
print(f"async_mode   : {metric.async_mode}")
print(f"strict_mode  : {metric.strict_mode}")

In [ ]:
# --------------------------------------------------------------------------
# Run the metric. A judge failure is caught and explained rather than left as
# a bare traceback, because "the judge could not be reached" and "the
# application scored badly" are completely different findings.
# --------------------------------------------------------------------------
try:
    metric.measure(test_case)
except Exception as exc:                      # noqa: BLE001 - diagnostic wrapper
    message = str(exc)
    print(f"METRIC EXECUTION FAILED: {type(exc).__name__}: {message[:600]}")
    if "api_key" in message.lower() or "authentication" in message.lower():
        print("  -> OPENAI_API_KEY is missing or rejected. This is the judge's key, "
              "not the application's.")
    elif "model" in message.lower() and "not" in message.lower():
        print(f"  -> The judge model {JUDGE_MODEL!r} was rejected. Check that your "
              f"OpenAI account can reach it, and that the installed DeepEval version "
              f"knows the id. Set DEEPEVAL_JUDGE_MODEL to change it.")
    elif "rate" in message.lower():
        print("  -> Rate limited by the judge provider. Re-run the cell.")
    raise

In [ ]:
# --------------------------------------------------------------------------
# Score, verdict, reason and debug output.
#
# Read `metric.is_successful()`, never the raw score: DeepEval metrics do not
# all point the same way. AnswerRelevancy and ToolCorrectness are "higher is
# better"; Bias and Hallucination are rates where lower is better; PIILeakage
# is a privacy score where 0.0 means maximum leakage. is_successful() applies
# the correct comparison for the metric.
# --------------------------------------------------------------------------
print(f"metric          : {type(metric).__name__}")
print(f"judge model     : {JUDGE_MODEL}")
print(f"threshold       : {metric.threshold}")
print(f"score           : {metric.score}")
print(f"PASS / FAIL     : {'PASS' if metric.is_successful() else 'FAIL'}")
print(f"judge cost (USD): {metric.evaluation_cost}")
print()
print("reason:")
print(textwrap.fill(str(metric.reason), width=96, subsequent_indent="  "))
print()
print("----- verbose judge log (debug) -----")
print(metric.verbose_logs or "(none - construct the metric with verbose_mode=True)")

## Limitations in a black-box acceptance test

1. **It only detects contradiction, not invention.** A claim about something the context
   is silent on is not a contradiction, so a plausible fabrication on an uncovered topic
   scores `0.0`. Breadth of `context` is what determines coverage.
2. **The score is per-document, not per-claim.** It is the fraction of context documents
   contradicted. One badly wrong sentence and five badly wrong sentences in the same
   document produce the same contribution.
3. **The ground truth here is the application's own corpus.** That is the right authority
   for "does the assistant follow *this bank's* policy", and the wrong one for "is this
   answer correct in the real world". The metric cannot tell the two apart.
4. **Long context strains the judge.** Five whole policy documents is a large prompt;
   judges get less reliable as context grows, and cost rises with it. Narrowing `context`
   to the governing policies raises reliability at the cost of coverage.
5. **This application makes unlabelled claims hard to emit in the first place.** An
   investigation's rationale must carry `C#`/`T#` evidence labels resolving to a real chunk
   or tool call, and synthesis fails server-side with `502 llm_response_invalid` if a label
   cannot be resolved. That is a structural defence, and it means a low hallucination score
   on the investigation surface partly reflects the API's validation rather than the
   model's restraint.